 # Grand Scraping File

You shall pick two dates, beginning and end, in the Japanese format, such as "2026-05-23", denoting the 23rd of May 2026.



The code will then create a folder:
scrandle_data


→ Inside this folder, it will create a folder for each date in the chosen timeframe, such as:
2026-05-23, 2026-05-24, ...



→→ Inside each date folder, it will create a CSV file containing the metadata, named:
meta.csv



→→→ This CSV file will contain the following information for each entry:

match_id (pair index)
side (left / right)
title
subtitle
price
rating
image_name
image_url




→→ Inside each date folder, it will create the 20 image files belonging to that date, named:
0_left.webp, 0_right.webp, 1_left.webp, 1_right.webp, ...



→→→ The naming convention ensures that:

i_left and i_right belong to the same comparison
Each date consists of 10 comparisons → 20 images in total


Each date folder will therefore contain:

20 image files
1 CSV file (meta.csv)

→ resulting in 21 files per date

In its current form, each date takes approximately 8 seconds to process.




# Ran correctly morning of 25-05-2026

In [2]:
#Imporing libraries
import requests
import pandas as pd
import os
from datetime import datetime, timedelta

In [3]:
# ------------------------------
# 1. Define date range
# ------------------------------
start_date = "2025-04-20"
end_date   = "2026-05-19"

start = datetime.strptime(start_date, "%Y-%m-%d")
end   = datetime.strptime(end_date, "%Y-%m-%d")

dates = []
current = start

while current <= end:
    dates.append(current.strftime("%Y-%m-%d"))
    current += timedelta(days=1)


#print(dates) #You are very welcome to check these and uncomment them.

In [4]:
# ------------------------------
# 2. Create base folder
# ------------------------------
base_folder = "../data/grand_scraper_folder/scrandle_data"

os.makedirs(base_folder, exist_ok=True)

In [ ]:
# ------------------------------
# 3. Loop over each date
# ------------------------------
headers = {"User-Agent": "Mozilla/5.0"}

for date in dates:

    print(f"\nProcessing {date}...")

    url = f"https://scrandle.com/history/{date}"

    try:
        response = requests.get(url, headers=headers)
        data = response.json()
    except:
        print(f"⚠️ Skipping {date} (no data available)")
        continue


    # ------------------------------
    # 4. Create folder for this date
    # ------------------------------
    date_folder = os.path.join(base_folder, date)
    os.makedirs(date_folder, exist_ok=True)


    # ------------------------------
    # 5. Prepare metadata rows
    # ------------------------------
    rows = []

    for i, (left, right) in enumerate(data["data"]):

        # ---------- LEFT ----------
        left_img_url = left["images_new"][0]
        left_filename = f"{i}_left.webp"

        rows.append({
            "date": date,
            "match_id": i,
            "side": "left",
            "title": left["title"].strip('"'),
            "subtitle": left["subtitle"],
            "price": left["price"],
            "rating": left["rating"],
            "image_name": left_filename,
            "image_url": left_img_url
        })

        # Download LEFT image
        try:
            img_data = requests.get(left_img_url).content
            with open(os.path.join(date_folder, left_filename), "wb") as f:
                f.write(img_data)
        except:
            print(f"⚠️ Failed LEFT image {i} on {date}")


        # ---------- RIGHT ----------
        right_img_url = right["images_new"][0]
        right_filename = f"{i}_right.webp"

        rows.append({
            "date": date,
            "match_id": i,
            "side": "right",
            "title": right["title"].strip('"'),
            "subtitle": right["subtitle"],
            "price": right["price"],
            "rating": right["rating"],
            "image_name": right_filename,
            "image_url": right_img_url
        })

        # Download RIGHT image
        try:
            img_data = requests.get(right_img_url).content
            with open(os.path.join(date_folder, right_filename), "wb") as f:
                f.write(img_data)
        except:
            print(f"⚠️ Failed RIGHT image {i} on {date}")


   # ------------------------------
    # 6. Save metadata CSV
    # ------------------------------
    df = pd.DataFrame(rows)

    csv_path = os.path.join(date_folder, "meta.csv")
    df.to_csv(csv_path, index=False)



#   print(dates) #You are very welcome to check these and uncomment them.



Processing 2025-04-20...

Processing 2025-04-21...

Processing 2025-04-22...

Processing 2025-04-23...

Processing 2025-04-24...

Processing 2025-04-25...

Processing 2025-04-26...

Processing 2025-04-27...

Processing 2025-04-28...

Processing 2025-04-29...

Processing 2025-04-30...

Processing 2025-05-01...
⚠️ Failed RIGHT image 3 on 2025-05-01

Processing 2025-05-02...

Processing 2025-05-03...

Processing 2025-05-04...

Processing 2025-05-05...

Processing 2025-05-06...

Processing 2025-05-07...

Processing 2025-05-08...


In [5]:
import os
import requests
import pandas as pd

headers = {"User-Agent": "Mozilla/5.0"}

for date in sorted(os.listdir(base_folder)):

    date_path = os.path.join(base_folder, date)

    if not os.path.isdir(date_path):
        continue

    print(f"Updating CSV for {date}")

    url = f"https://scrandle.com/history/{date}"

    try:
        response = requests.get(url, headers=headers)
        data = response.json()
    except:
        print(f"⚠️ Skipping {date} (no data)")
        continue

    rows = []

    for i, (left, right) in enumerate(data["data"]):

        # -------- LEFT --------
        rows.append({
            "date": date,
            "match_id": i,
            "side": "left",

            "id": left["id"],
            "url": left["url"],
            "price": left["price"],
            "rating": left["rating"],
            "year": left["year"],
            "title": left["title"].strip('"'),
            "subtitle": left["subtitle"],
            "club": left["club"],
            "country": left["country"],
            "image_name": f"{i}_left.webp",
            "image_url": left["images_new"][0],
            "rating_source": left["rating_source"],
            "image_position": left["image_position"],
            "elo": left["elo"],
        })

        # -------- RIGHT --------
        rows.append({
            "date": date,
            "match_id": i,
            "side": "right",

            "id": right["id"],
            "url": right["url"],
            "price": right["price"],
            "rating": right["rating"],
            "year": right["year"],
            "title": right["title"].strip('"'),
            "subtitle": right["subtitle"],
            "club": right["club"],
            "country": right["country"],
            "image_name": f"{i}_right.webp",
            "image_url": right["images_new"][0],
            "rating_source": right["rating_source"],
            "image_position": right["image_position"],
            "elo": right["elo"],
        })

    # Overwrite CSV only
    df = pd.DataFrame(rows)
    csv_path = os.path.join(date_path, "meta.csv")
    df.to_csv(csv_path, index=False)

print("✅ All CSV files updated with full metadata")


Updating CSV for 2025-04-20
Updating CSV for 2025-04-21
Updating CSV for 2025-04-22
Updating CSV for 2025-04-23
Updating CSV for 2025-04-24
Updating CSV for 2025-04-25
Updating CSV for 2025-04-26
Updating CSV for 2025-04-27
Updating CSV for 2025-04-28
Updating CSV for 2025-04-29
Updating CSV for 2025-04-30
Updating CSV for 2025-05-01
Updating CSV for 2025-05-02
Updating CSV for 2025-05-03
Updating CSV for 2025-05-04
Updating CSV for 2025-05-05
Updating CSV for 2025-05-06
Updating CSV for 2025-05-07
Updating CSV for 2025-05-08
Updating CSV for 2025-05-09
Updating CSV for 2025-05-10
Updating CSV for 2025-05-11
Updating CSV for 2025-05-12
Updating CSV for 2025-05-13
Updating CSV for 2025-05-14
Updating CSV for 2025-05-15
Updating CSV for 2025-05-16
Updating CSV for 2025-05-17
Updating CSV for 2025-05-18
Updating CSV for 2025-05-19
Updating CSV for 2025-05-20
Updating CSV for 2025-05-21
Updating CSV for 2025-05-22
Updating CSV for 2025-05-23
Updating CSV for 2025-05-24
Updating CSV for 202